In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

import torch
from torch import nn

In [2]:
DATA = Path("../../data/")
DATA_PROCESSED = DATA / "processed"
DATA_PREDICTIONS = DATA / "predictions"

train_returns = pd.read_csv(DATA_PROCESSED / "train_returns.csv", index_col=0, parse_dates=True)
valid_returns = pd.read_csv(DATA_PROCESSED / "valid_returns.csv", index_col=0, parse_dates=True)
test_returns = pd.read_csv(DATA_PROCESSED / "test_returns.csv", index_col=0, parse_dates=True)

ASSETS = train_returns.columns.tolist()
NUM_ASSETS = len(ASSETS)

In [3]:
for name, df in [
    ("Train", train_returns),
    ("Valid", valid_returns),
    ("Test", test_returns)
]:
    assert df.columns.tolist() == ASSETS
    assert np.isfinite(df.to_numpy()).all()

    print(
        f"{name:12s} | "
        f"{df.index.min().date()} - {df.index.max().date()} | "
        f"{df.shape}"
    )

Train        | 2015-05-14 - 2017-12-29 | (646, 30)
Valid        | 2018-01-02 - 2018-12-28 | (243, 30)
Test         | 2019-01-02 - 2024-08-30 | (1376, 30)


In [4]:
class BaselineModel:
    def __init__(self, num_assets: int, window_size: int = 1, window_type: str = "rolling"):
        assert window_size > 0
        assert window_type in ("rolling", "expanding")

        self.num_assets = num_assets
        self.window_size = window_size
        self.window_type = window_type

    def forward(self, x: np.ndarray) -> np.ndarray:
        # x: (days, num_assets)
        # out: (days - window_size + 1, num_assets)
        assert x.ndim == 2
        assert x.shape[1] == self.num_assets
        assert x.shape[0] >= self.window_size

        out = []
        for i in range(x.shape[0] - self.window_size + 1):
            end = i + self.window_size
            inputs = x[i:end] if self.window_type == "rolling" else x[:end]
            out.append(np.mean(inputs, axis=0))

        return np.stack(out, axis=0)

    def __call__(self, x: np.ndarray) -> np.ndarray:
        return self.forward(x)

# Training and Evaluation

In [5]:
# No training is required for the historical mean baseline

In [6]:
def evaluate(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    assert y_true.shape == y_pred.shape
    assert y_true.ndim == 2
    assert np.isfinite(y_true).all()
    assert np.isfinite(y_pred).all()

    errors = y_true - y_pred

    mse = np.mean(np.power(errors, 2))
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(errors))
    directional_accuracy = np.mean(np.sign(y_true) == np.sign(y_pred))

    true_df = pd.DataFrame(y_true)
    pred_df = pd.DataFrame(y_pred)
    ic = pred_df.corrwith(true_df, axis=1, method="pearson")
    rank_ic = pred_df.corrwith(true_df, axis=1, method="spearman")

    return {
        "MSE": mse,
        "RMSE": rmse,
        "MAE": mae,
        "Directional Accuracy": directional_accuracy,
        "IC": ic.mean(),
        "Rank IC": rank_ic.mean(),
        "ICIR": ic.mean() / ic.std(),
        "Rank ICIR": rank_ic.mean() / rank_ic.std()
    }

# Hyperparameter tuning

In [7]:
best_ic = -float("inf")
best_config = None

for window_type in ("rolling", "expanding"):
    for window_size in (1, 2, 4, 8, 16, 32, 64, 128, 7, 14, 21, 28):
        model = BaselineModel(num_assets=NUM_ASSETS, window_size=window_size, window_type=window_type)

        X_valid = valid_returns.iloc[:-1].to_numpy()
        y_valid_true = valid_returns.iloc[window_size:].to_numpy()
        y_valid_pred = model(X_valid)

        valid_metrics = evaluate(y_valid_true, y_valid_pred)

        if valid_metrics["IC"] > best_ic:
            best_ic = valid_metrics["IC"]

            best_config = {
                "window_type": window_type,
                "window_size": window_size,
                "best_metric": float(best_ic)
            }

display(best_config)

{'window_type': 'expanding',
 'window_size': 2,
 'best_metric': 0.0016195205972236722}

# Out-Of-Sample

In [8]:
model = BaselineModel(NUM_ASSETS, window_size=best_config["window_size"], window_type=best_config["window_type"])

X_test = test_returns.iloc[:-1].to_numpy()
y_test_true = test_returns.iloc[model.window_size:].to_numpy()
y_test_pred = model(X_test)
test_metrics = evaluate(y_test_true, y_test_pred)
test_metrics

{'MSE': np.float64(0.0006638034897809608),
 'RMSE': np.float64(0.025764384133546852),
 'MAE': np.float64(0.017906899164008033),
 'Directional Accuracy': np.float64(0.46523532265890344),
 'IC': np.float64(-0.00752153010153765),
 'Rank IC': np.float64(-0.030956424520867276),
 'ICIR': np.float64(-0.028009414035879403),
 'Rank ICIR': np.float64(-0.1269027948312572)}

## Saving

In [9]:
test_predictions = pd.DataFrame(y_test_pred, index=test_returns.index[model.window_size:], columns=test_returns.columns)
test_predictions.to_csv(DATA_PREDICTIONS / "baseline_oos_predictions.csv")
display(test_predictions)

,600169.SH,000049.SZ,600428.SH,002064.SZ,000501.SZ,603000.SH,000712.SZ,600580.SH,600017.SH,603355.SH,...,600718.SH,600507.SH,002583.SZ,601718.SH,600835.SH,002268.SZ,002092.SZ,600098.SH,002340.SZ,300001.SZ
date,,,,,,,,,,,,,,,,,,,,,
2019-01-04,0.004474,0.009040,0.004614,0.008336,-0.002446,0.012969,0.013135,0.002378,-0.005315,0.007497,...,-2.961844e-02,0.003152,-0.019750,0.011914,0.015083,0.013902,-0.001415,-0.006203,0.001302,-0.016258
2019-01-07,0.010390,0.008082,0.011206,0.008687,0.005793,0.021106,0.039515,0.010538,0.002561,0.010345,...,-9.319360e-03,0.007414,-0.003052,0.016691,0.016949,0.018528,0.005695,-0.001750,0.010392,-0.000619
2019-01-08,0.015401,0.006657,0.013613,0.008841,0.007716,0.018404,0.035776,0.016750,0.007317,0.009225,...,-3.644260e-03,0.016345,0.009555,0.021038,0.018267,0.022242,0.006014,-0.000868,0.011582,0.002540
2019-01-09,0.012321,0.003831,0.012057,0.005690,0.006378,0.014978,0.029963,0.013994,0.011488,0.008008,...,-3.971735e-03,0.011572,0.006910,0.016282,0.017299,0.017178,0.003703,0.002851,0.007773,0.000788
2019-01-10,0.012378,0.009009,0.008598,0.004742,0.006507,0.015238,0.027985,0.010427,0.009573,0.007641,...,-4.047307e-03,0.010906,0.007395,0.012189,0.012418,0.015258,0.004247,0.001795,0.006060,-0.005790
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-08-26,0.000068,0.000692,0.000826,0.000742,0.000032,0.001229,0.000517,0.000781,0.000187,0.000658,...,-8.898543e-06,0.000110,-0.000188,-0.000036,0.000121,0.000184,-0.000139,0.000314,0.000652,0.000462
2024-08-27,0.000080,0.000724,0.000836,0.000740,0.000039,0.001230,0.000487,0.000795,0.000184,0.000654,...,-5.045482e-06,0.000112,-0.000182,-0.000030,0.000122,0.000184,-0.000129,0.000311,0.000668,0.000467
2024-08-28,0.000072,0.000679,0.000838,0.000744,0.000042,0.001213,0.000498,0.000767,0.000184,0.000641,...,-1.555614e-05,0.000110,-0.000202,-0.000033,0.000114,0.000172,-0.000139,0.000311,0.000655,0.000430
